# SpiderNet data loading and MI-dimension-selection tutorial (PerturbFISH)

This notebook is a step-by-step tutorial for loading and preprocessing the PerturbFISH spatial transcriptomics data for SpiderNet, as well as running the MI-dimension-selection procedure on the processed data.

1. **Basic user inputs**
2. **PerturbFISH-specific SpiderNet input assumptions**
3. **Build the base config**
4. **Preview the LR-correlation density before choosing `lr_corr_threshold`**
5. **Run the full unified loader**
6. **Inspect outputs**
7. **Save PerturbFISH-specific sample metadata**
8. **Run MI dimension selection on the processed outputs**


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import scanpy as sc

# If the notebook is run from the project root, this is usually enough.
sys.path.append(str(Path.cwd()))

from SpiderNet.dataloading_unified import (
    prepare_processed_bundle_unified,
    preview_lr_corr_distribution,
)
from SpiderNet.utils import (
    get_default_cellchat_db,
    get_default_scseqcomm_db,
)


## Step 1. Basic user inputs

Edit the variables in this section first.


In [ ]:
from run_benchmarks import DATA_ROOT, PROCESSED_ROOT
OUTPUT_DIR = PROCESSED_ROOT

SPECIES = "human"  ## Species used for ligand-receptor database loading. Options are "human" or "mouse".

SAMPLE_COL = "sample_name"  ## The sample-defining column in adata.obs. If this column is absent, the original PerturbFISH loader falls back to a single sample named "Sample1".
CELL_CLASS_COL = "celltype2"  ## The cell type annotation column in adata.obs.
PYG_EXTRA_OBS_FIELDS = {}  ## Extra adata.obs fields to copy into each PyG sample object. Leave empty unless you want to propagate additional metadata.

N_HVG = 1000  ## Kept for consistency with PerturbFISH_modeltraining; the original PerturbFISH loader accepts this parameter but does not apply HVG subsetting.
N_HVG_LR = 2000  ## Kept for consistency with PerturbFISH_modeltraining; the original PerturbFISH loader accepts this parameter but does not apply HVG subsetting.
NUM_NEIGHBORS = 10  ## Number of spatial neighbors. In PerturbFISH_modeltraining, NUM_NEIGHBORS is set to 10.

# Set this only after running the preview section below
LR_CORR_THRESHOLD = None  ## Set this to the desired ligand-receptor correlation threshold after inspecting the preview plot in Step 4. In the original PerturbFISH loader, the final LR-correlation filter uses 0.5.


## Step 2. PerturbFISH-specific SpiderNet input

This notebook follows the PerturbFISH data convention used in the original
`dataloading_PerturbFISH` script and the `PerturbFISH_modeltraining` notebook:

- expression is read directly from `X`
- spatial coordinates are stored in `obsm["spatial"]`
- cell types are defined by `obs["celltype2"]`
- samples are defined by `obs["sample_name"]`
- if `obs["sample_name"]` is absent, the loader falls back to a single sample `"Sample1"`
- the original PerturbFISH loader does **not** run `normalize_total` / `log1p`
- the original PerturbFISH loader does **not** apply HVG subsetting before saving the processed bundle

Because this convention is specific to the PerturbFISH data, these settings are
kept out of the main user-input block above.


In [ ]:
CELLCHAT_DB = get_default_cellchat_db(species=SPECIES)
SCSEQCOMM_DB = get_default_scseqcomm_db(species=SPECIES)

DATA_REPRESENTATION_CONFIG = {
    "expression_source": {"kind": "X", "name": None},
    "normalize_strategy": "never",
    "log1p": False,
    "remove_zero_count_cells": False,
    "spatial_source": {"kind": "obsm", "key": "spatial"},
    "apply_hvg_selection": False,
}


## Step 3. Build the base config

This config contains the common loading settings for the PerturbFISH dataset.
`lr_corr_threshold` is intentionally left unset until after the preview step.


In [ ]:
base_config = {
    "data_path_main": DATA_ROOT,
    "output_dir": OUTPUT_DIR,
    "ligand_receptor_filedir_cellchatdb": CELLCHAT_DB,
    "ligand_receptor_filedir_scSeqComm": SCSEQCOMM_DB,

    "sample_col": SAMPLE_COL,
    "cell_class_col": CELL_CLASS_COL,
    "pyg_obs_fields": PYG_EXTRA_OBS_FIELDS,

    "n_hvg": N_HVG,
    "n_hvg_lr": N_HVG_LR,
    "num_neighbors": NUM_NEIGHBORS,

    # Keep unset until after the preview
    "lr_corr_threshold": LR_CORR_THRESHOLD,
}

base_config.update(DATA_REPRESENTATION_CONFIG)

print("CellChat DB:", CELLCHAT_DB)
print("scSeqComm DB:", SCSEQCOMM_DB)
print("Base output dir:", OUTPUT_DIR)

pd.Series({
    "data_path_main": str(base_config["data_path_main"]),
    "output_dir": str(base_config["output_dir"]),
    "sample_col": base_config["sample_col"],
    "cell_class_col": base_config["cell_class_col"],
    "n_hvg": base_config["n_hvg"],
    "n_hvg_lr": base_config["n_hvg_lr"],
    "num_neighbors": base_config["num_neighbors"],
    "expression_source": str(base_config["expression_source"]),
    "normalize_strategy": base_config["normalize_strategy"],
    "spatial_source": str(base_config["spatial_source"]),
    "apply_hvg_selection": base_config["apply_hvg_selection"],
}).to_frame("value")


## Step 4. Preview the LR-correlation density

Run this cell **before** setting `LR_CORR_THRESHOLD`.

This preview only runs the minimal preprocessing needed to reach the LR-correlation stage.
It does **not** run the full `prepare_processed_bundle_unified(...)` pipeline, and it only shows
the density plot needed for choosing `lr_corr_threshold`.


In [ ]:
preview = preview_lr_corr_distribution(base_config, show_plot=True)

if preview.get("plot_path") is not None:
    print("Preview plot saved to:", preview["plot_path"])


## Step 5. Set `lr_corr_threshold` and run the full loader

After inspecting the density plot above, set `LR_CORR_THRESHOLD` to the desired value.
In the original PerturbFISH loader, the LR-correlation filter uses `0.5`, so that is a
reasonable default starting point here.


In [ ]:
LR_CORR_THRESHOLD = 0.5

In [ ]:
if LR_CORR_THRESHOLD is None:
    raise ValueError(
        "Please set LR_CORR_THRESHOLD in Step 1 after inspecting the preview plot, "
        "then rerun Step 3 and this cell."
    )

config = dict(base_config)
config["lr_corr_threshold"] = float(LR_CORR_THRESHOLD)

bundle = prepare_processed_bundle_unified(config)


## Step 6. Inspect the processed outputs

In [ ]:
print("Output dir:", bundle["output_dir"])
print("Number of batches:", len(bundle["SpiderNet_data_pyg_list"]))
print("Number of retained LR pairs:", len(bundle["LR_list"]))
print("Number of training genes:", len(bundle["genenames_train"]))
print("Batch labels:", bundle["batch_cell_unique"])

bundle["adata"]


## Step 7. Save PerturbFISH-specific sample metadata

This section summarizes one row per sample from the processed PerturbFISH AnnData and
saves it as `metadata_sample.csv` in the processed output directory.


In [ ]:
obs = bundle["adata"].obs.copy()

metadata_sample = (
    obs.groupby(SAMPLE_COL, dropna=False)
      .agg(
          num_cells=(SAMPLE_COL, "size"),
          n_cell_types=(CELL_CLASS_COL, pd.Series.nunique),
      )
      .reset_index()
)

celltype_summary = (
    obs.groupby(SAMPLE_COL)[CELL_CLASS_COL]
      .apply(lambda x: "; ".join(sorted(pd.Series(x).dropna().astype(str).unique())))
      .rename("cell_types")
      .reset_index()
)

metadata_sample = (
    metadata_sample
    .merge(celltype_summary, on=SAMPLE_COL, how="left")
    .sort_values(SAMPLE_COL, kind="stable")
    .reset_index(drop=True)
)

metadata_sample.to_csv(Path(bundle["output_dir"]) / "metadata_sample.csv", index=False)
metadata_sample


## Step 8. Run MI dimension selection on the processed outputs

This section reuses the processed files that were just written to `bundle["output_dir"]`.
It does **not** rerun data loading or preprocessing.

By default, the MI-dimension-selection thresholds are chosen automatically based on the
number of retained LR pairs. You can leave the optional overrides below as `None` unless
you want to tune the heuristic manually.


In [ ]:
from IPython.display import display
import pandas as pd

from SpiderNet.io import load_processed_data
from SpiderNet.MI_dimension_selection import run_mi_dimension_selection

# Optional advanced overrides for MI dimension selection.
# Leave these as None to use the default automatic heuristic.
MI_DIM_LR_SPEARCOR_THRESHOLD = None
MI_DIM_MIN_CLIQUE_SIZE = None
MI_DIM_JACCARD_THRESHOLD = None
MI_DIM_SHOW_HEATMAP = True

processed = load_processed_data(bundle["output_dir"])

print("Processed directory:", bundle["output_dir"])
print("Number of batches:", len(processed.spidernet_data))
print("Number of retained LR pairs:", len(processed.lr_list))
print("Number of training genes:", len(processed.genenames_train))

mi_dim_results = run_mi_dimension_selection(
    processed=processed,
    lr_list=processed.lr_list,
    output_dir=bundle["output_dir"],
    lr_spearcor_threshold=MI_DIM_LR_SPEARCOR_THRESHOLD,
    min_clique_size=MI_DIM_MIN_CLIQUE_SIZE,
    jaccard_thr=MI_DIM_JACCARD_THRESHOLD,
    show=MI_DIM_SHOW_HEATMAP,
)

mi_dim_summary_df = pd.DataFrame(
    [
        {
            "recommended_dim_envir": mi_dim_results["recommended_dim_envir"],
            "num_merged_subsets": mi_dim_results["num_merged_subsets"],
            "subset_sizes": mi_dim_results["subset_sizes"],
            "num_lr_pairs": mi_dim_results["num_lr_pairs"],
            "num_graph_edges": mi_dim_results["num_graph_edges"],
            "num_maximal_cliques_filtered": mi_dim_results["num_maximal_cliques_filtered"],
            "effective_min_clique_size": mi_dim_results["effective_min_clique_size"],
            "num_unassigned_lr_pairs": mi_dim_results["num_unassigned_lr_pairs"],
            "lr_spearcor_threshold": mi_dim_results["lr_spearcor_threshold"],
            "jaccard_thr": mi_dim_results["jaccard_thr"],
            "warning": mi_dim_results["warning"],
        }
    ]
)

display(mi_dim_summary_df)

print(f"Recommended dim_envir: {mi_dim_results['recommended_dim_envir']}")
print(f"Heatmap saved to: {mi_dim_results['heatmap_path']}")
print(f"Summary JSON saved to: {mi_dim_results['summary_path']}")
print(f"Subset summary CSV saved to: {mi_dim_results['subset_summary_path']}")
